In [ ]:
from pyspark.sql import SparkSession
import kagglehub
from pyspark.sql.functions import col, when, month, hour

spark = SparkSession.builder.appName("FlightDelay_GBT").getOrCreate()

path = kagglehub.dataset_download("williamparker20/flight-ontime-reporting-with-weather")
df = spark.read.option("recursiveFileLookup", "true").csv(path, header=True, inferSchema=True)

cols_needed = ["DepDelayMinutes", "Time", "Origin", "Dest", "Carrier",
               "Temperature", "Wind_Speed", "Precipitation",
               "Wind_Gust", "Ice_Accretion_3hr"]
df_clean = df.select(cols_needed).na.fill(0.0, ["Wind_Gust", "Ice_Accretion_3hr"]).dropna()


df_prepared = df_clean.withColumn("Label", when(col("DepDelayMinutes") > 15, 1.0).otherwise(0.0)) \
                      .withColumn("Month", month(col("Time"))) \
                      .withColumn("Hour", hour(col("Time")))

train_data, test_data = df_prepared.randomSplit([0.8, 0.2], seed=42)

print("Analiza rozkładu klas w train_data:")
class_counts = train_data.groupBy("Label").count()
class_counts.show()

total_rows = train_data.count()
print(f"Łączna liczba wierszy w train_data: {total_rows}")

print("Proporcje klas:")
class_proportions = class_counts.withColumn("Proportion", col("count") / total_rows)
class_proportions.show()

100%|██████████| 157M/157M [00:05<00:00, 27.7MB/s]

Extracting files...


Analiza rozkładu klas w train_data:
+-----+-------+
|Label|  count|
+-----+-------+
|  0.0|8928570|
|  1.0|2410147|
+-----+-------+

Łączna liczba wierszy w train_data: 11338717
Proporcje klas:
+-----+-------+-------------------+
|Label|  count|         Proportion|
+-----+-------+-------------------+
|  0.0|8928570|  0.787440942392336|
|  1.0|2410147|0.21255905760766408|
+-----+-------+-------------------+



## Ważenie klas do GBT

In [ ]:
from pyspark.sql.functions import lit

# 1. Obliczanie wag klas

count_0 = class_counts.filter(col("Label") == 0.0).select("count").collect()[0][0]
count_1 = class_counts.filter(col("Label") == 1.0).select("count").collect()[0][0]

weight_0 = total_rows / (2 * count_0)
weight_1 = total_rows / (2 * count_1)

print(f"Obliczona waga dla klasy 0.0 (nieopóźnione): {weight_0:.4f}")
print(f"Obliczona waga dla klasy 1.0 (opóźnione): {weight_1:.4f}")

# 2. Dodawanie kolumny 'class_weight' do train_data
train_data_weighted = train_data.withColumn("class_weight",
                                           when(col("Label") == 0.0, lit(weight_0))
                                           .otherwise(lit(weight_1)))

print("Kolumna 'class_weight' dodana do train_data_weighted.")
train_data_weighted.select("Label", "class_weight").show(5)

# 3. Zaktualizowanie obiektu GBTClassifier o weightCol

if 'gbt' not in locals():
    from pyspark.ml.classification import GBTClassifier
    gbt = GBTClassifier(labelCol="Label",
                        featuresCol="features",
                        maxIter=30,
                        maxDepth=5,
                        maxBins=500)

gbt_weighted = gbt.setWeightCol("class_weight")

print("GBTClassifier zaktualizowany o 'weightCol'.")

Obliczona waga dla klasy 0.0 (nieopóźnione): 0.6350
Obliczona waga dla klasy 1.0 (opóźnione): 2.3523
Kolumna 'class_weight' dodana do train_data_weighted.
+-----+------------------+
|Label|      class_weight|
+-----+------------------+
|  0.0|0.6349682535949206|
|  0.0|0.6349682535949206|
|  0.0|0.6349682535949206|
|  0.0|0.6349682535949206|
|  0.0|0.6349682535949206|
+-----+------------------+
only showing top 5 rows
GBTClassifier zaktualizowany o 'weightCol'.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.functions import col
from pyspark.ml.classification import GBTClassifier


indexer_origin = StringIndexer(inputCol="Origin", outputCol="OriginIndex", handleInvalid="keep")
indexer_dest = StringIndexer(inputCol="Dest", outputCol="DestIndex", handleInvalid="keep")
indexer_carrier = StringIndexer(inputCol="Carrier", outputCol="CarrierIndex", handleInvalid="keep")


assembler = VectorAssembler(
    inputCols=["OriginIndex", "DestIndex", "CarrierIndex", "Month", "Hour",
               "Temperature", "Wind_Speed", "Wind_Gust", "Precipitation", "Ice_Accretion_3hr"],
    outputCol="features"
)


if 'gbt' not in locals():
    gbt = GBTClassifier(labelCol="Label",
                        featuresCol="features",
                        maxIter=30,
                        maxDepth=5,
                        maxBins=500)
if 'gbt_weighted' not in locals():
    gbt_weighted = gbt.setWeightCol("class_weight")

print("Tworzenie nowego potoku z ważeniem klas...")
pipeline_weighted = Pipeline(stages=[indexer_origin, indexer_dest, indexer_carrier, assembler, gbt_weighted])

print("Rozpoczynam trenowanie Gradient Boosted Trees z ważeniem (To może potrwać)...\n")
model_weighted = pipeline_weighted.fit(train_data_weighted)
print("Model GBT z ważeniem gotowy!")

predictions_weighted = model_weighted.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="Label")
auc_weighted = evaluator.evaluate(predictions_weighted)

auc_previous = 0.7048

print(f"\n>>> WYNIK AUC (GBT z ważeniem): {auc_weighted:.4f} <<<")
print(f"Poprzedni WYNIK AUC (GBT bez ważenia): {auc_previous:.4f}")

importances_weighted = model_weighted.stages[-1].featureImportances
print("\nWażność cech (z ważeniem):\n")
print(importances_weighted)


Tworzenie nowego potoku z ważeniem klas...
Rozpoczynam trenowanie Gradient Boosted Trees z ważeniem (To może potrwać)...

Model GBT z ważeniem gotowy!

>>> WYNIK AUC (GBT z ważeniem): 0.7049 <<<
Poprzedni WYNIK AUC (GBT bez ważenia): 0.7048

Ważność cech (z ważeniem):

(10,[0,1,2,3,4,5,6,7,8],[0.13515688189299213,0.0950891198878013,0.16416643727471453,0.23669305696937235,0.17216771731115804,0.09766792228345694,0.01275004924996168,0.01972652216564406,0.06658229296489894])
